# Django, ModelForms

## Introduction to ModelForms
---

In this lesson, you'll learn how to **create forms directly from your Django models**.

ModelForms eliminate duplication by automatically generating form fields based on your model's field definitions.

1. [Why ModelForms](#Why-ModelForms),
    - [DRY Principle](#DRY-Principle),
    - [ModelForm vs Form](#ModelForm-vs-Form),
2. [Creating a ModelForm](#Creating-a-ModelForm),
    - [Basic Structure](#Basic-Structure),
    - [The Meta Class](#The-Meta-Class),
3. [Controlling Fields](#Controlling-Fields),
    - [fields vs exclude](#fields-vs-exclude),
    - [Field Customization](#Field-Customization),
4. [Saving Data](#Saving-Data),
    - [The save() Method](#The-save()-Method),
    - [commit=False Pattern](#commit=False-Pattern),
5. [Editing Existing Objects](#Editing-Existing-Objects),
    - [The instance Parameter](#The-instance-Parameter),
    - [Update Views](#Update-Views),
6. [🧠 EXERCISE 🧠, Create a Book Form](#🧠-EXERCISE-🧠,-Create-a-Book-Form),
7. [🧠 EXERCISE 🧠, Build an Edit View](#🧠-EXERCISE-🧠,-Build-an-Edit-View).

<br>

## Why ModelForms

---

### DRY Principle

---

Consider this model and its corresponding form:

<br>

**The Problem - Duplication:**

In [ ]:
# models.py - Model definition
from django.db import models


class Book(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    isbn = models.CharField(max_length=13)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    published_date = models.DateField()
    description = models.TextField(blank=True)

In [ ]:
# forms.py - Manual form definition (BAD - duplicates model fields!)
from django import forms


class BookForm(forms.Form):  # BAD: Duplicating model fields
    title = forms.CharField(max_length=200)
    author = forms.CharField(max_length=100)
    isbn = forms.CharField(max_length=13)
    price = forms.DecimalField(max_digits=10, decimal_places=2)
    published_date = forms.DateField()
    description = forms.CharField(required=False, widget=forms.Textarea)

<br>

**The problems with this approach:**

| Issue | Consequence |
|-------|-------------|
| Duplicated field definitions | Changes need to be made in two places |
| Manual validation matching | Easy to get out of sync |
| Manual save logic | You write `book.title = form.cleaned_data['title']` for every field |
| Error-prone | Typos, forgotten fields, inconsistent constraints |

<br>

### ModelForm vs Form

---

**ModelForm** solves this by generating form fields from your model:

<br>

| Feature | `forms.Form` | `forms.ModelForm` |
|---------|--------------|-------------------|
| Field source | Manually defined | Auto-generated from model |
| Validation | Manual | Inherits model constraints |
| Saving | Manual `Model.objects.create()` | Built-in `form.save()` |
| Use case | Any data collection | CRUD operations on models |

<br>

**Use ModelForm when:**
- You're creating/editing model instances
- Form fields match model fields closely

**Use regular Form when:**
- Data doesn't map to a model (e.g., search, contact forms)
- Form structure differs significantly from any model

<br>

## Creating a ModelForm

---

### Basic Structure

---

A ModelForm is a form class that inherits from `forms.ModelForm`:

In [ ]:
# catalog/forms.py

from django import forms
from .models import Book


class BookForm(forms.ModelForm):  # GOOD: Inherits from ModelForm
    """Form for creating and editing books."""
    
    class Meta:
        model = Book
        fields = ['title', 'author', 'isbn', 'price', 'published_date', 'description']

<br>

That's it! Django automatically:
- Creates form fields matching the model fields
- Applies the same constraints (`max_length`, `blank`, etc.)
- Provides a `save()` method that creates/updates the model instance

<br>

### The Meta Class

---

The `Meta` inner class configures how the ModelForm behaves:

<br>

| Attribute | Purpose | Example |
|-----------|---------|--------|
| `model` | Which model to use | `model = Book` |
| `fields` | Which fields to include | `fields = ['title', 'author']` |
| `exclude` | Which fields to exclude | `exclude = ['created_at']` |
| `widgets` | Custom widgets for fields | `widgets = {'description': Textarea}` |
| `labels` | Custom labels | `labels = {'isbn': 'ISBN Number'}` |
| `help_texts` | Custom help texts | `help_texts = {'isbn': '13 digits'}` |
| `error_messages` | Custom error messages | `error_messages = {...}` |

In [ ]:
# Example: Fully configured Meta class

from django import forms
from .models import Book


class BookForm(forms.ModelForm):
    """Form for creating and editing books."""
    
    class Meta:
        model = Book
        fields = ['title', 'author', 'isbn', 'price', 'published_date', 'description']
        
        widgets = {
            'published_date': forms.DateInput(
                attrs={'type': 'date'}  # HTML5 date picker
            ),
            'description': forms.Textarea(
                attrs={'rows': 4, 'placeholder': 'Enter book description...'}
            ),
        }
        
        labels = {
            'isbn': 'ISBN Number',
            'published_date': 'Publication Date',
        }
        
        help_texts = {
            'isbn': 'Enter the 13-digit ISBN.',
            'price': 'Price in USD.',
        }

<br>

## Controlling Fields

---

### fields vs exclude

---

You **must** specify which fields to include in the form. There are three approaches:

<br>

**1. Explicit field list (recommended):**

In [ ]:
class BookForm(forms.ModelForm):
    class Meta:
        model = Book
        fields = ['title', 'author', 'price']  # Only these fields

<br>

**2. All fields:**

In [ ]:
class BookForm(forms.ModelForm):
    class Meta:
        model = Book
        fields = '__all__'  # All model fields (be careful!)

<br>

**3. Exclude specific fields:**

In [ ]:
class BookForm(forms.ModelForm):
    class Meta:
        model = Book
        exclude = ['created_at', 'updated_at']  # All except these

<br>

**Security Warning:** Always use explicit `fields` when possible!

| Approach | Security | Recommendation |
|----------|----------|----------------|
| `fields = [...]` | ✅ Safe | **Recommended** - explicit is better |
| `fields = '__all__'` | ⚠️ Risky | Avoid - exposes all fields |
| `exclude = [...]` | ⚠️ Risky | Avoid - new fields auto-included |

Using `exclude` or `__all__` can accidentally expose sensitive fields when you add new fields to the model.

<br>

### Field Customization

---

You can override or add fields to a ModelForm just like a regular form:

In [ ]:
from django import forms
from .models import Book


class BookForm(forms.ModelForm):
    """Form with customized and additional fields."""
    
    # Override an existing field with custom validation
    price = forms.DecimalField(
        max_digits=10,
        decimal_places=2,
        min_value=0.01,
        help_text='Price must be at least $0.01'
    )
    
    # Add a field that doesn't exist in the model
    confirm_price = forms.DecimalField(
        label='Confirm Price',
        help_text='Re-enter the price to confirm.'
    )
    
    class Meta:
        model = Book
        fields = ['title', 'author', 'price', 'description']

<br>

**Field override rules:**
- If you define a field with the same name as a model field, your definition takes precedence
- You can add fields that don't exist in the model (handle them in `clean()` or the view)
- Model field constraints still apply during save (database-level)

<br>

## Saving Data

---

### The save() Method

---

ModelForm provides a `save()` method that creates or updates the model instance:

In [ ]:
# Creating a new book

from django.shortcuts import render, redirect
from .forms import BookForm


def create_book(request):
    """View to create a new book."""
    
    if request.method == 'POST':
        form = BookForm(request.POST)
        
        if form.is_valid():
            # save() creates the Book instance and saves to database
            book = form.save()
            
            # book is now a saved Book instance with a primary key
            print(f"Created book: {book.title} (ID: {book.pk})")
            
            return redirect('book_detail', pk=book.pk)
    else:
        form = BookForm()
    
    return render(request, 'catalog/book_form.html', {'form': form})

<br>

**What `save()` does:**

1. Creates a model instance with the form data
2. Calls `instance.save()` to persist to the database
3. Returns the saved instance

<br>

### commit=False Pattern

---

Sometimes you need to modify the instance before saving. Use `commit=False`:

In [ ]:
# Adding data not in the form before saving

def create_book(request):
    """View to create a book with additional data."""
    
    if request.method == 'POST':
        form = BookForm(request.POST)
        
        if form.is_valid():
            # commit=False creates instance but doesn't save to database
            book = form.save(commit=False)
            
            # Now you can modify the instance
            book.created_by = request.user  # Add the current user
            book.status = 'draft'           # Set default status
            
            # Now save to database
            book.save()
            
            return redirect('book_detail', pk=book.pk)
    else:
        form = BookForm()
    
    return render(request, 'catalog/book_form.html', {'form': form})

<br>

**Common uses for `commit=False`:**

| Scenario | Example |
|----------|--------|
| Add user reference | `book.created_by = request.user` |
| Set timestamps | `book.created_at = timezone.now()` |
| Set computed values | `book.slug = slugify(book.title)` |
| Set defaults | `book.status = 'pending'` |

<br>

**Important:** If your model has ManyToMany fields, you must call `save_m2m()` after using `commit=False`:

In [ ]:
# Handling ManyToMany fields with commit=False

def create_book_with_categories(request):
    """View handling a form with ManyToMany field."""
    
    if request.method == 'POST':
        form = BookForm(request.POST)
        
        if form.is_valid():
            book = form.save(commit=False)
            book.created_by = request.user
            book.save()  # Save the book first
            
            # Now save ManyToMany relationships (e.g., categories)
            form.save_m2m()
            
            return redirect('book_detail', pk=book.pk)
    else:
        form = BookForm()
    
    return render(request, 'catalog/book_form.html', {'form': form})

<br>

## Editing Existing Objects

---

### The instance Parameter

---

To edit an existing object, pass it as the `instance` parameter:

In [ ]:
from django.shortcuts import render, redirect, get_object_or_404
from .models import Book
from .forms import BookForm


def edit_book(request, pk):
    """View to edit an existing book."""
    
    # Get the existing book or return 404
    book = get_object_or_404(Book, pk=pk)
    
    if request.method == 'POST':
        # Bind POST data AND the existing instance
        form = BookForm(request.POST, instance=book)
        
        if form.is_valid():
            # save() updates the existing instance
            form.save()
            return redirect('book_detail', pk=book.pk)
    else:
        # Pre-fill form with existing data
        form = BookForm(instance=book)
    
    return render(request, 'catalog/book_form.html', {
        'form': form,
        'book': book  # Pass book for template context
    })

<br>

**How `instance` works:**

| Without `instance` | With `instance` |
|--------------------|----------------|
| Form fields are empty | Form fields are pre-filled |
| `save()` creates new object | `save()` updates existing object |
| `INSERT` SQL | `UPDATE` SQL |

<br>

### Update Views

---

A complete pattern for create and update views:

In [ ]:
# catalog/views.py - Complete CRUD views

from django.shortcuts import render, redirect, get_object_or_404
from django.contrib import messages
from .models import Book
from .forms import BookForm


def book_create(request):
    """Create a new book."""
    
    if request.method == 'POST':
        form = BookForm(request.POST)
        if form.is_valid():
            book = form.save()
            messages.success(request, f'Book "{book.title}" created successfully!')
            return redirect('book_detail', pk=book.pk)
    else:
        form = BookForm()
    
    return render(request, 'catalog/book_form.html', {
        'form': form,
        'title': 'Add New Book'
    })


def book_update(request, pk):
    """Update an existing book."""
    
    book = get_object_or_404(Book, pk=pk)
    
    if request.method == 'POST':
        form = BookForm(request.POST, instance=book)
        if form.is_valid():
            form.save()
            messages.success(request, f'Book "{book.title}" updated successfully!')
            return redirect('book_detail', pk=book.pk)
    else:
        form = BookForm(instance=book)
    
    return render(request, 'catalog/book_form.html', {
        'form': form,
        'book': book,
        'title': f'Edit: {book.title}'
    })

In [ ]:
# templates/catalog/book_form.html - Reusable template

template_form = """
{% extends 'base.html' %}

{% block title %}{{ title }}{% endblock %}

{% block content %}
<h1>{{ title }}</h1>

<form method="post" novalidate>
    {% csrf_token %}
    
    {% for field in form %}
        <div class="form-group">
            {{ field.label_tag }}
            {{ field }}
            {% if field.help_text %}
                <small>{{ field.help_text }}</small>
            {% endif %}
            {% if field.errors %}
                <span class="error">{{ field.errors.0 }}</span>
            {% endif %}
        </div>
    {% endfor %}
    
    <button type="submit">
        {% if book %}Update{% else %}Create{% endif %} Book
    </button>
    
    {% if book %}
        <a href="{% url 'book_detail' pk=book.pk %}">Cancel</a>
    {% endif %}
</form>
{% endblock %}
"""
print(template_form)

In [ ]:
# catalog/urls.py

from django.urls import path
from . import views

urlpatterns = [
    path('books/new/', views.book_create, name='book_create'),
    path('books/<int:pk>/edit/', views.book_update, name='book_update'),
]

<br>

## Combining Create and Update

---

You can combine create and update into a single view:

In [ ]:
# Combined create/update view

def book_form(request, pk=None):
    """Create or update a book depending on whether pk is provided."""
    
    # Get existing book or None for new book
    if pk:
        book = get_object_or_404(Book, pk=pk)
        title = f'Edit: {book.title}'
    else:
        book = None
        title = 'Add New Book'
    
    if request.method == 'POST':
        form = BookForm(request.POST, instance=book)
        if form.is_valid():
            saved_book = form.save()
            action = 'updated' if pk else 'created'
            messages.success(request, f'Book {action} successfully!')
            return redirect('book_detail', pk=saved_book.pk)
    else:
        form = BookForm(instance=book)
    
    return render(request, 'catalog/book_form.html', {
        'form': form,
        'book': book,
        'title': title
    })

In [ ]:
# URL patterns for combined view

urlpatterns = [
    path('books/new/', views.book_form, name='book_create'),
    path('books/<int:pk>/edit/', views.book_form, name='book_update'),
]

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **ModelForm** | Form generated from model - follows DRY principle |
| **Meta class** | Configure model, fields, widgets, labels |
| **fields** | Always use explicit list for security |
| **exclude** | Avoid - can accidentally expose new fields |
| **save()** | Creates or updates model instance |
| **commit=False** | Create instance without saving - modify first |
| **save_m2m()** | Required after commit=False with ManyToMany fields |
| **instance** | Pass existing object to edit instead of create |

<br>

### 🧠 EXERCISE 🧠, Create a Book Form

---

Create a ModelForm for managing books in your bookstore:

1. First, ensure you have a `Book` model in `catalog/models.py` with:
   - `title` (CharField, max 200)
   - `author` (CharField, max 100)
   - `isbn` (CharField, max 13)
   - `price` (DecimalField)
   - `description` (TextField, optional)
   - `published_date` (DateField)

2. Create a `BookForm` ModelForm in `catalog/forms.py`:
   - Include all fields except any auto-generated ones
   - Use a date input widget for `published_date`
   - Add custom labels and help text

3. Create a view `book_create` that displays and processes the form

4. Create the template and URL pattern

5. Run migrations and test creating a book

<details>
    <summary>▶️ Solution</summary>
    
**1. catalog/models.py:**

```python
from django.db import models


class Book(models.Model):
    """Model representing a book in the catalog."""
    
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    isbn = models.CharField(max_length=13, unique=True)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    description = models.TextField(blank=True)
    published_date = models.DateField()
    
    def __str__(self) -> str:
        return f"{self.title} by {self.author}"
```

**2. catalog/forms.py:**

```python
from django import forms
from .models import Book


class BookForm(forms.ModelForm):
    """Form for creating and editing books."""
    
    class Meta:
        model = Book
        fields = ['title', 'author', 'isbn', 'price', 'description', 'published_date']
        
        widgets = {
            'published_date': forms.DateInput(attrs={'type': 'date'}),
            'description': forms.Textarea(attrs={'rows': 4}),
        }
        
        labels = {
            'isbn': 'ISBN',
            'published_date': 'Publication Date',
        }
        
        help_texts = {
            'isbn': 'Enter the 13-digit ISBN number.',
            'price': 'Price in USD.',
        }
```

**3. catalog/views.py:**

```python
from django.shortcuts import render, redirect
from .forms import BookForm


def book_create(request):
    if request.method == 'POST':
        form = BookForm(request.POST)
        if form.is_valid():
            book = form.save()
            print(f"Created: {book.title}")
            return redirect('book_create')  # Or to a detail page
    else:
        form = BookForm()
    
    return render(request, 'catalog/book_form.html', {'form': form})
```

**4. templates/catalog/book_form.html:**

```html
<!DOCTYPE html>
<html>
<head>
    <title>Add Book</title>
</head>
<body>
    <h1>Add New Book</h1>
    
    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">Save Book</button>
    </form>
</body>
</html>
```

**catalog/urls.py:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('books/new/', views.book_create, name='book_create'),
]
```

**5. Run migrations:**

```bash
python manage.py makemigrations
python manage.py migrate
python manage.py runserver
# Visit http://127.0.0.1:8000/books/new/
```
</details>

<br>

### 🧠 EXERCISE 🧠, Build an Edit View

---

Extend the previous exercise to support editing:

1. Create a `book_update` view that:
   - Takes a `pk` parameter to identify the book
   - Pre-fills the form with existing data
   - Updates the book on valid submission

2. Add a URL pattern for the edit view

3. Modify the template to show "Edit" or "Create" based on context

4. Create a book in the admin or via the create form, then edit it

<details>
    <summary>▶️ Solution</summary>
    
**1. catalog/views.py (add update view):**

```python
from django.shortcuts import render, redirect, get_object_or_404
from .models import Book
from .forms import BookForm


def book_create(request):
    if request.method == 'POST':
        form = BookForm(request.POST)
        if form.is_valid():
            book = form.save()
            return redirect('catalog:book_update', pk=book.pk)
    else:
        form = BookForm()
    
    return render(request, 'catalog/book_form.html', {
        'form': form,
        'title': 'Add New Book'
    })


def book_update(request, pk):
    book = get_object_or_404(Book, pk=pk)
    
    if request.method == 'POST':
        form = BookForm(request.POST, instance=book)
        if form.is_valid():
            form.save()
            return redirect('catalog:book_update', pk=book.pk)
    else:
        form = BookForm(instance=book)
    
    return render(request, 'catalog/book_form.html', {
        'form': form,
        'book': book,
        'title': f'Edit: {book.title}'
    })
```

**2. catalog/urls.py:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('books/new/', views.book_create, name='book_create'),
    path('books/<int:pk>/edit/', views.book_update, name='book_update'),
]
```

**3. templates/catalog/book_form.html:**

```html
<!DOCTYPE html>
<html>
<head>
    <title>{{ title }}</title>
</head>
<body>
    <h1>{{ title }}</h1>
    
    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">
            {% if book %}Update{% else %}Create{% endif %} Book
        </button>
    </form>
    
    {% if book %}
        <p>Editing book ID: {{ book.pk }}</p>
    {% endif %}
</body>
</html>
```

**4. Test:**

```bash
python manage.py runserver
# Create: http://127.0.0.1:8000/books/new/
# Edit: http://127.0.0.1:8000/books/1/edit/ (after creating book with ID 1)
```
</details>

---